# CrisisInbox GRPO Training

Train a small LLM to triage crisis inbox messages using Group Relative Policy Optimization.

**What this does:**
1. Loads pre-generated episode data (inbox snapshots at decision points)
2. For each prompt, the model generates an action (which message to handle + response)
3. A reward function scores the action based on urgency, deadline, drift adaptation
4. GRPO updates the model to prefer higher-reward actions

**GPU profiles:**
- **T4 / free Colab**: Qwen2.5-0.5B, 2048 ctx, 4-bit — runs in ~30 min
- **H100 / A100**: Qwen2.5-3B, 4096 ctx, 4-bit — better quality, ~20 min

In [ ]:
# Install dependencies
!pip install unsloth trl transformers datasets accelerate peft -q
!pip install huggingface_hub -q

# Download episode data
# Option 1: From HF dataset (recommended)
# Option 2: From GitHub repo
# Option 3: Generate locally with `python generate_episodes.py -n 100`

import os
if not os.path.exists("episodes.json"):
    print("Downloading episodes.json from GitHub...")
    !wget -q --show-progress https://raw.githubusercontent.com/eptan/crisis-inbox/main/episodes.json
    if not os.path.exists("episodes.json"):
        print("ERROR: Download failed. Upload episodes.json manually or generate with:")
        print("  !git clone https://github.com/eptan/crisis-inbox.git && cd crisis-inbox && python generate_episodes.py -n 100")
else:
    print("episodes.json already exists, skipping download")

print("Setup complete")

In [ ]:
# === GPU PROFILE ===
# Change this one variable to switch between T4 and H100 configs.
# Everything else adapts automatically.

import torch

if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name} ({vram_gb:.0f} GB)")
else:
    vram_gb = 0
    print("No GPU detected — config will default to smallest profile")

# Auto-select profile based on VRAM, or override manually
if vram_gb >= 40:  # H100, A100
    PROFILE = "h100"
    MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"
    MAX_SEQ_LENGTH = 4096
    MAX_PROMPT_LENGTH = 3584
    MAX_COMPLETION_LENGTH = 512
    BATCH_SIZE = 4
    GRAD_ACCUM = 2
    NUM_GENERATIONS = 8
elif vram_gb >= 14:  # T4, L4
    PROFILE = "t4"
    MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct"
    MAX_SEQ_LENGTH = 2048
    MAX_PROMPT_LENGTH = 1792
    MAX_COMPLETION_LENGTH = 256
    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    NUM_GENERATIONS = 4
else:
    PROFILE = "cpu"
    MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct"
    MAX_SEQ_LENGTH = 2048
    MAX_PROMPT_LENGTH = 1792
    MAX_COMPLETION_LENGTH = 256
    BATCH_SIZE = 1
    GRAD_ACCUM = 8
    NUM_GENERATIONS = 2

print(f"Profile: {PROFILE} | Model: {MODEL_NAME} | Context: {MAX_SEQ_LENGTH}")

In [ ]:
import json
import re
import random
from datasets import Dataset

# Load episodes
with open("episodes.json") as f:
    episodes = json.load(f)

# Check format — old format has 'messages'/'tasks', new format has 'decision_points'
if episodes and "decision_points" not in episodes[0]:
    old_keys = list(episodes[0].keys())
    raise ValueError(
        f"episodes.json is in the old format (keys: {old_keys}).\n"
        f"Regenerate with: python generate_episodes.py -n 100\n"
        f"The old format used 'messages'/'tasks'/'schema_events'; "
        f"the notebook requires 'decision_points' from generate_episodes.py."
    )

# Flatten to individual training prompts
prompts = []
for ep in episodes:
    for dp in ep["decision_points"]:
        prompts.append({
            "prompt": dp["prompt"],
            "hour": dp["hour"],
            "visible_count": dp["visible_count"],
            "episode_id": ep["episode_id"],
            "seed": ep["seed"],
            "drift_events": ep["drift_events"],
            "superseded": ep.get("superseded_messages", {}),
            "messages": dp["visible_messages"],
        })

if not prompts:
    raise ValueError("No decision_points found in episodes; cannot train.")

print(f"Loaded {len(episodes)} episodes -> {len(prompts)} training prompts")
print(f"Average {len(prompts)/len(episodes):.1f} decision points per episode")

## Reward Function

Scores agent actions based on:
- **Urgency base** (critical=10, high=5, medium=3, low=1)
- **Deadline timing** (early=bonus, late=penalty)
- **Drift adaptation** (+50% for handling policy-change messages)
- **Stale info penalty** (-50% for acting on superseded messages)
- **Response quality** (penalty for short/empty responses)

In [ ]:
def score_action(completion: str, prompt_data: dict) -> float:
    """
    Score a model completion against the inbox state.
    
    The model should output: respond_to_message(msg_id, "response text")
    We parse the message_id and response, then score based on the reward function.
    """
    messages = prompt_data["messages"]
    hour = prompt_data["hour"]
    superseded = prompt_data.get("superseded", {})
    
    # Parse the model output for message_id
    msg_id = None
    response_text = ""
    
    # Try to parse respond_to_message(msg_id, response)
    match = re.search(r'respond_to_message\s*\(\s*["\']?(msg_\d+)["\']?\s*,\s*["\'](.+?)["\']', completion, re.DOTALL)
    if match:
        msg_id = match.group(1)
        response_text = match.group(2)
    else:
        # Try simpler format: just a message ID mentioned
        id_match = re.search(r'(msg_\d+)', completion)
        if id_match:
            msg_id = id_match.group(1)
            # No explicit response text — penalize via quality check below
            response_text = ""
    
    if not msg_id:
        return -1.0  # couldn't parse any action
    
    # Find the message in the inbox
    target_msg = None
    for msg in messages:
        if msg["id"] == msg_id:
            target_msg = msg
            break
    
    if target_msg is None:
        return -0.5  # referenced a message not in inbox
    
    # Base reward by urgency
    urgency_rewards = {"critical": 10.0, "high": 5.0, "medium": 3.0, "low": 1.0}
    reward = urgency_rewards.get(target_msg["urgency"], 1.0)
    
    # Deadline timing
    deadline = target_msg.get("deadline_hours")
    if deadline is not None:
        if hour <= deadline:
            time_remaining_frac = (deadline - hour) / max(deadline, 1.0)
            reward *= 1.0 + 0.5 * time_remaining_frac
        else:
            reward *= 0.25  # late penalty
    
    # Response quality
    if len(response_text.strip()) < 10:
        reward *= 0.5
    
    # Drift adaptation bonus
    if target_msg.get("drift_flag"):
        reward *= 1.5
    
    # Stale info penalty
    if target_msg["id"] in superseded:
        reward *= 0.5
    
    # Penalize choosing low-urgency when unhandled critical messages exist
    unhandled_critical = any(
        m["urgency"] == "critical" and not m.get("handled") and not m.get("superseded")
        for m in messages
    )
    if unhandled_critical and target_msg["urgency"] in ("low", "medium"):
        reward *= 0.3
    
    return round(reward, 2)


# Test the reward function
test_data = prompts[0]
print("Testing reward function on first decision point:")
print(f"  Hour: {test_data['hour']}, Messages: {test_data['visible_count']}")

# Simulate good action (pick critical message)
critical_msgs = [m for m in test_data["messages"] if m["urgency"] == "critical"]
if critical_msgs:
    good_action = f'respond_to_message("{critical_msgs[0]["id"]}", "Acknowledged. Evacuating immediately with documents and medication.")'
    good_score = score_action(good_action, test_data)
    print(f"  Good action (critical msg): {good_score:.2f} pts")

# Simulate bad action (pick low-urgency message)
low_msgs = [m for m in test_data["messages"] if m["urgency"] == "low"]
if low_msgs:
    bad_action = f'respond_to_message("{low_msgs[0]["id"]}", "ok")'
    bad_score = score_action(bad_action, test_data)
    print(f"  Bad action (low msg, short response): {bad_score:.2f} pts")

# Simulate unparseable action
junk_score = score_action("I think we should do something", test_data)
print(f"  Unparseable action: {junk_score:.2f} pts")

## Load Model & Configure GRPO

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Add LoRA adapters — bigger r for bigger models
lora_r = 32 if PROFILE == "h100" else 16

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_r,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=lora_r,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
print(f"Model loaded: {MODEL_NAME} | LoRA r={lora_r} | ctx={MAX_SEQ_LENGTH}")

In [ ]:
# Build the training dataset
# Each row needs a "prompt" field formatted as chat messages
train_data = []
for p in prompts:
    train_data.append({
        "prompt": [
            {"role": "user", "content": p["prompt"]},
        ],
        # Store metadata for reward calculation (not used by trainer directly)
        "_hour": p["hour"],
        "_episode_id": p["episode_id"],
    })

# Shuffle and split
random.seed(42)
random.shuffle(train_data)

dataset = Dataset.from_list(train_data)
print(f"Training dataset: {len(dataset)} prompts")
print(f"Sample prompt length: {len(train_data[0]['prompt'][0]['content'])} chars")

## GRPO Training Loop

The reward function scores each completion by:
1. Parsing which message the model chose to handle
2. Checking urgency, deadline timing, drift flags
3. Penalizing bad choices (low-urgency when critical exists, stale info)

In [ ]:
from trl import GRPOConfig, GRPOTrainer

# Build a lookup from (episode_id, hour) -> prompt metadata for reward scoring
prompt_lookup = {}
for p in prompts:
    key = (p["episode_id"], p["hour"])
    prompt_lookup[key] = p


def reward_fn(prompts, completions, _episode_id, _hour, **kwargs):
    """
    GRPO reward function. Scores each completion against its inbox state.

    TRL passes extra dataset columns as keyword arguments, so _episode_id and
    _hour come directly from the dataset — no need to reverse-lookup from text.
    """
    rewards = []
    for completion, ep_id, hour in zip(completions, _episode_id, _hour):
        key = (ep_id, hour)
        prompt_data = prompt_lookup.get(key)

        if prompt_data is None:
            rewards.append(0.0)
            continue

        if isinstance(completion, list):
            comp_text = completion[-1]["content"] if completion else ""
        else:
            comp_text = str(completion)

        score = score_action(comp_text, prompt_data)
        rewards.append(score)

    return rewards


print(f"Prompt lookup: {len(prompt_lookup)} unique keys (expect {len(prompts)})")

# GRPO training config — all values from GPU profile
training_args = GRPOConfig(
    output_dir="crisisinbox-grpo-output",
    num_train_epochs=3,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=5e-6,
    max_completion_length=MAX_COMPLETION_LENGTH,
    max_prompt_length=MAX_PROMPT_LENGTH,
    num_generations=NUM_GENERATIONS,
    logging_steps=10,
    save_steps=100,
    report_to="none",
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=dataset,
)

print(f"Trainer configured — batch={BATCH_SIZE}, gen={NUM_GENERATIONS}, prompt≤{MAX_PROMPT_LENGTH}tok")

In [ ]:
# Train!
trainer.train()
print("Training complete")

## Evaluate Trained Model

Sample prompts and check whether the model picks high-urgency messages and produces well-formatted actions.

In [ ]:
# Evaluate on a few test prompts
FastLanguageModel.for_inference(model)

eval_prompts = random.sample(prompts, min(10, len(prompts)))
total_score = 0

print(f"=== Trained Model Evaluation ({MODEL_NAME}) ===\n")
for p in eval_prompts:
    messages = [{"role": "user", "content": p["prompt"]}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")

    with torch.no_grad():
        output = model.generate(inputs, max_new_tokens=MAX_COMPLETION_LENGTH, temperature=0.7, do_sample=True)

    completion = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True)
    score = score_action(completion, p)
    total_score += score

    # Show a summary
    msg_match = re.search(r'(msg_\d+)', completion)
    chosen_id = msg_match.group(1) if msg_match else "none"
    chosen_msg = next((m for m in p["messages"] if m["id"] == chosen_id), None)
    urgency = chosen_msg["urgency"] if chosen_msg else "?"

    print(f"Hour {p['hour']:5.1f} | Chose: {chosen_id} ({urgency:8s}) | Score: {score:+.1f}")

print(f"\nAverage score: {total_score / len(eval_prompts):.2f}")

In [ ]:
# Save the trained model
model.save_pretrained("crisisinbox-grpo-trained")
tokenizer.save_pretrained("crisisinbox-grpo-trained")
print("Model saved to crisisinbox-grpo-trained/")